In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

Leakage / GroupKFold — коли звичайний CV бреше через спільні групи

- **Group leakage**: один і той самий `user_id` / `device_id` / `session` потрапляє і в train, і в test → модель «впізнає» юзера, метрики завищені.
- **GroupKFold** тримає всю групу лише в одному фолді: або train, або test.
- Типові групи: `user_id`, `client_id`, `shop_id`, `patient_id`.
- Для часу окремо: `TimeSeriesSplit` (не shuffle минулого/майбутнього).
- Якщо `KFold` score сильно вищий за `GroupKFold` — майже напевно є group leakage.

In [2]:
def leakage_groupkfold_check(
    df,
    feature_cols,
    target_col='target',
    group_col='user_id',
    model=None,
    n_splits=5,
    scoring='roc_auc',
    random_state=42
):
    """
    Порівнює звичайний KFold і GroupKFold на одних і тих самих даних.

    Параметри
    ---------
    df : pandas.DataFrame
        Датафрейм з фічами, таргетом і групою (наприклад user_id).

    feature_cols : sequence of str
        Список фіч для моделі.

    target_col : str, за замовчуванням 'target'
        Бінарний / класифікаційний таргет.

    group_col : str, за замовчуванням 'user_id'
        Колонка групи, яку не можна розривати між train/test.

    model : estimator або None
        Якщо None → GradientBoostingClassifier.

    n_splits : int, за замовчуванням 5
        Кількість фолдів.

    scoring : str, за замовчуванням 'roc_auc'
        Метрика для cross_val_score.

    random_state : int, за замовчуванням 42
        Seed для KFold (GroupKFold детермінований по групах).

    Повертає
    --------
    pandas.DataFrame
        Два рядки (KFold / GroupKFold):
        - CV, Mean Score, Std Score
        - Overlap Users (скільки унікальних груп перетинаються train∩test у середньому)
        - N Groups, N Rows

    Приклади
    --------
    >>> leakage_groupkfold_check(df, feature_cols=['gross_summ', 'avg_session_time'])
    """
    data = df[list(feature_cols) + [target_col, group_col]].dropna().copy()
    X = data[list(feature_cols)]
    y = data[target_col]
    groups = data[group_col]

    if model is None:
        model = GradientBoostingClassifier(random_state=random_state)

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    gkf = GroupKFold(n_splits=n_splits)

    scores_kfold = cross_val_score(model, X, y, cv=kfold, scoring=scoring)
    scores_group = cross_val_score(model, X, y, cv=gkf, groups=groups, scoring=scoring)

    overlap_kfold = _mean_group_overlap(kfold.split(X), groups)
    overlap_group = _mean_group_overlap(gkf.split(X, y, groups), groups)

    return pd.DataFrame([
        {
            'CV': 'KFold',
            'Mean Score': round(scores_kfold.mean(), 4),
            'Std Score': round(scores_kfold.std(), 4),
            'Overlap Users': round(overlap_kfold, 2),
            'N Groups': groups.nunique(),
            'N Rows': len(data),
            'Scoring': scoring,
        },
        {
            'CV': 'GroupKFold',
            'Mean Score': round(scores_group.mean(), 4),
            'Std Score': round(scores_group.std(), 4),
            'Overlap Users': round(overlap_group, 2),
            'N Groups': groups.nunique(),
            'N Rows': len(data),
            'Scoring': scoring,
        },
    ])


def _mean_group_overlap(splits, groups):
    overlaps = []
    groups = np.asarray(groups)
    for train_idx, test_idx in splits:
        train_g = set(groups[train_idx])
        test_g = set(groups[test_idx])
        overlaps.append(len(train_g & test_g))
    return float(np.mean(overlaps))

### демо

Кілька рядків на одного `user_id`. У фічах є «відбиток» юзера → звичайний KFold підглядає, GroupKFold — ні.

In [4]:
rng = np.random.default_rng(42)
n_users = 200
rows_per_user = 5

rows = []
for user_id in range(n_users):
    user_base = rng.normal(0, 1)
    user_target = int(user_base + rng.normal(0, 0.5) > 0)
    for _ in range(rows_per_user):
        rows.append({
            'user_id': user_id,
            'gross_summ': user_base * 10 + rng.normal(50, 3),
            'avg_session_time': user_base * 2 + rng.normal(12, 0.8),
            'target': user_target,
        })

df = pd.DataFrame(rows)
df.head()

,user_id,gross_summ,avg_session_time,target
0,0,55.298524,13.361886,0
1,0,47.194065,11.567691,0
2,0,53.430692,12.356440,0
3,0,52.996767,11.926999,0
4,0,55.685365,13.231668,0


In [5]:
leakage_groupkfold_check(
    df,
    feature_cols=['gross_summ', 'avg_session_time'],
    target_col='target',
    group_col='user_id',
    scoring='roc_auc'
)

,CV,Mean Score,Std Score,Overlap Users,N Groups,N Rows,Scoring
0,KFold,0.8816,0.0303,131.2,200,1000,roc_auc
1,GroupKFold,0.8873,0.0387,0.0,200,1000,roc_auc


Очікувано: у `KFold` буде `Overlap Users > 0` і вищий score; у `GroupKFold` overlap = 0 і більш чесна оцінка.